In [1]:
import random
random.seed(1000)

def random_sample_filter_by_length(input_file, output_file, sample_size=2000000, max_length=100):
    """
    从大文件中随机采样指定数量的行，并过滤掉长度超过max_length的SMILES
    使用水库采样算法，内存高效
    """
    print("正在统计符合条件的SMILES行数...")

    # 第一次遍历：统计符合长度条件的总行数
    total_valid_lines = 0
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            smiles = line.strip()
            if smiles and len(smiles) <= max_length:
                total_valid_lines += 1

    print(f"文件中总行数（长度≤{max_length}）: {total_valid_lines:,}")

    if sample_size > total_valid_lines:
        print(f"警告: 采样数量({sample_size:,})大于符合条件的行数({total_valid_lines:,})")
        print(f"将采样所有符合条件的行: {total_valid_lines:,}")
        sample_size = total_valid_lines

    if sample_size == 0:
        print("错误: 没有符合条件的SMILES行")
        return

    # 使用水库采样算法选择随机行号
    print(f"正在从符合条件的行中随机选择 {sample_size:,} 行...")

    # 生成随机索引
    indices_to_keep = set(random.sample(range(total_valid_lines), sample_size))

    print("正在写入采样结果...")

    # 第二次遍历，只选长度≤100且索引匹配的行
    valid_line_idx = 0
    selected_count = 0

    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(output_file, 'w', encoding='utf-8') as f_out:

        for idx, line in enumerate(f_in):
            smiles = line.strip()

            # 只处理长度≤100的SMILES
            if smiles and len(smiles) <= max_length:
                if valid_line_idx in indices_to_keep:
                    f_out.write(smiles + '\n')
                    selected_count += 1
                valid_line_idx += 1

            # 进度提示
            if (idx + 1) % 1000000 == 0:
                print(f"已扫描 {idx+1:,} 行，找到 {valid_line_idx:,} 条符合条件的SMILES...")

    print(f"\n完成！")
    print(f"输出文件: {output_file}")
    print(f"输出行数: {selected_count:,}")
    print(f"过滤掉的长度>100的SMILES: {idx+1 - valid_line_idx:,} 行")


# 使用

input_file = "sampled_5M.txt"  # 替换为你的输入文件名
output_file = "sampled_2M.txt"  # 输出文件名

try:
    random_sample_filter_by_length(input_file, output_file, sample_size=2000000, max_length=100)
except FileNotFoundError:
    print(f"错误: 找不到文件 {input_file}")
except Exception as e:
    print(f"发生错误: {e}")

正在统计符合条件的SMILES行数...
文件中总行数（长度≤100）: 5,000,000
正在从符合条件的行中随机选择 2,000,000 行...
正在写入采样结果...
已扫描 1,000,000 行，找到 1,000,000 条符合条件的SMILES...
已扫描 2,000,000 行，找到 2,000,000 条符合条件的SMILES...
已扫描 3,000,000 行，找到 3,000,000 条符合条件的SMILES...
已扫描 4,000,000 行，找到 4,000,000 条符合条件的SMILES...
已扫描 5,000,000 行，找到 5,000,000 条符合条件的SMILES...

完成！
输出文件: sampled_2M.txt
输出行数: 2,000,000
过滤掉的长度>100的SMILES: 0 行


In [2]:
from rdkit import Chem
from tqdm import tqdm

# 输入输出文件
input_file = "sampled_2M.txt"  # 你的500万行文件
output_file = "sampled_2M_standardized.txt"
error_file = "errors_2M.log"

print("开始标准化SMILES...")
print("="*50)

# 统计总行数
print("统计文件行数...")
with open(input_file, 'r', encoding='utf-8') as f:
    total_lines = sum(1 for _ in f)
print(f"总行数: {total_lines:,}")

# 处理数据
valid_count = 0
invalid_count = 0

with open(input_file, 'r', encoding='utf-8') as f_in, \
     open(output_file, 'w', encoding='utf-8') as f_out, \
     open(error_file, 'w', encoding='utf-8') as f_err:

    # 使用进度条
    with tqdm(total=total_lines, desc="标准化SMILES", unit="行") as pbar:
        for line in f_in:
            line = line.strip()
            if not line:
                pbar.update(1)
                continue

            try:
                # 转换为分子
                mol = Chem.MolFromSmiles(line)

                if mol is None:
                    invalid_count += 1
                    f_err.write(f"Invalid: {line}\n")
                else:
                    # 标准化SMILES
                    std_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
                    f_out.write(std_smiles + '\n')
                    valid_count += 1

            except Exception as e:
                invalid_count += 1
                f_err.write(f"Error: {line} - {str(e)}\n")

            pbar.update(1)

# 打印结果
print("\n" + "="*50)
print("处理完成！")
print(f"总行数: {total_lines:,}")
print(f"有效SMILES: {valid_count:,} ({valid_count/total_lines*100:.2f}%)")
print(f"无效SMILES: {invalid_count:,} ({invalid_count/total_lines*100:.2f}%)")
print(f"输出文件: {output_file}")
print(f"错误日志: {error_file}")

开始标准化SMILES...
统计文件行数...
总行数: 2,000,000


标准化SMILES: 100%|██████████| 2000000/2000000 [06:43<00:00, 4952.72行/s]



处理完成！
总行数: 2,000,000
有效SMILES: 2,000,000 (100.00%)
无效SMILES: 0 (0.00%)
输出文件: sampled_2M_standardized.txt
错误日志: errors_2M.log


In [3]:
def split_smiles_file(input_file, output_file_99, output_file_1, split_ratio=0.99):
    """
    将SMILES文件按比例分成两个文件
    split_ratio: 第一个文件的比例（默认0.99即99%）
    """
    print(f"正在读取文件: {input_file}")

    # 读取所有行
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    total_lines = len(lines)
    print(f"总行数: {total_lines:,}")

    # 计算分割点
    split_point = int(total_lines * split_ratio)

    # 随机打乱顺序（可选，如果需要随机分配）
    # 如果希望保持原始顺序，注释掉下面这行
    random.shuffle(lines)

    # 分割数据
    part_99 = lines[:split_point]
    part_1 = lines[split_point:]

    print(f"99%部分: {len(part_99):,} 行")
    print(f"1%部分: {len(part_1):,} 行")

    # 写入文件
    print(f"正在写入99%文件: {output_file_99}")
    with open(output_file_99, 'w', encoding='utf-8') as f:
        f.writelines(part_99)

    print(f"正在写入1%文件: {output_file_1}")
    with open(output_file_1, 'w', encoding='utf-8') as f:
        f.writelines(part_1)

    print("完成！")
    print(f"文件1 (99%): {output_file_99} - {len(part_99):,} 行")
    print(f"文件2 (1%): {output_file_1} - {len(part_1):,} 行")

# 使用示例
if __name__ == "__main__":
    input_file = "sampled_2M_standardized.txt"  # 你的500万行文件
    output_file_99 = "sampled_1.98M.txt"  # 99%部分（约495万行）
    output_file_1 = "sampled_0.02M.txt"     # 1%部分（约5万行）

    split_smiles_file(input_file, output_file_99, output_file_1, split_ratio=0.99)

正在读取文件: sampled_2M_standardized.txt
总行数: 2,000,000
99%部分: 1,980,000 行
1%部分: 20,000 行
正在写入99%文件: sampled_1.98M.txt
正在写入1%文件: sampled_0.02M.txt
完成！
文件1 (99%): sampled_1.98M.txt - 1,980,000 行
文件2 (1%): sampled_0.02M.txt - 20,000 行
